# VLM cross-modal unlearning — Stage 1 runner

Thin runner: every cell calls one script. All outputs go to `/content/drive/MyDrive/vlm_unlearn/`.
Every job is **resumable** — after a disconnect, re-run cells 1–3 and then the job cell you were on.
See `README.md` for order and runtimes.

## 1. Setup (run every session)

In [ ]:
# --- set once ---------------------------------------------------------------
REPO_URL = "https://github.com/<you>/VLMUnlearn.git"   # private repo: https://<token>@github.com/<you>/VLMUnlearn.git
BRANCH = "main"
MODEL = "qwen2vl"          # Phase A: "qwen2vl"   |   Phase B: "llava"
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["VLMU_ROOT"] = "/content/drive/MyDrive/vlm_unlearn"   # checkpoints, logs, results
os.environ["VLMU_LOCAL_CACHE"] = "/content/vlmu_cache"            # fast local copy of processed data
os.environ["HF_HOME"] = "/content/hf_home"                        # base model weights (local disk)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs(os.environ["VLMU_ROOT"], exist_ok=True)

In [ ]:
import os
if not os.path.exists("/content/VLMUnlearn"):
    !git clone -b {BRANCH} {REPO_URL} /content/VLMUnlearn
else:
    !git -C /content/VLMUnlearn pull --ff-only
%cd /content/VLMUnlearn
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.is_available())"

## 2. Data (once; CPU is fine) — ~10 min

In [ ]:
!python scripts/prepare_data.py --config configs/data.yaml

## 3. Smoke test (once per model, before spending credits) — ~10–15 min
Runs every job below with `--debug` (~20 samples each) into `vlm_unlearn/debug/`. The final sanity step is *expected* to report a failed gate (the debug vanilla trains for 6 steps).

In [ ]:
!python scripts/smoke_test.py --model {MODEL}

## 4. Pretrained baseline (no fine-tuning: measures priors)

In [ ]:
!python scripts/eval_pretrained.py --config configs/{MODEL}/vanilla.yaml

## 5. Vanilla fine-tune (both forms) + full eval + gate (≥80% paired forget MC in both modalities)
If the gate fails, unlearning is blocked — stop and report.

In [ ]:
!python scripts/finetune_vanilla.py --config configs/{MODEL}/vanilla.yaml
!cat /content/drive/MyDrive/vlm_unlearn/runs/{MODEL}/vanilla/gate.json

## 6. Unlearning grid (18 runs: 3 methods × 2 modalities × 3 lr, seed 0)
Skips finished runs; re-run this cell after a disconnect. Failed runs are logged in `results/run_log.csv` and the grid continues.

In [ ]:
!python scripts/run_grid.py --config configs/{MODEL}/grid.yaml

## 7. Matched-forgetting selection + seed-1 reruns + full evals

In [ ]:
!python scripts/select_and_seed.py --config configs/{MODEL}/grid.yaml

## 8. Tables, figure, verdict + sanity checks

In [ ]:
!python scripts/aggregate.py --config configs/{MODEL}/grid.yaml
!python scripts/sanity.py --config configs/{MODEL}/grid.yaml
!cat /content/drive/MyDrive/vlm_unlearn/results/{MODEL}/verdict.json

## 9. (optional) push results to the repo
Results are always on Drive; this copies the mirrored `results/` into git.

In [ ]:
!git add results && git -c user.email=colab@local -c user.name=colab commit -m "results: {MODEL}" && git push